In [6]:
# ==== Oscilloscope Analyzer — dialogs, hover, sliders+number inputs, Phase modes (Abs/Rel/TF) with -180..180 wrap ====

# Backend（元コードを残していますが、Plotly描画なので実質未使用です）
try:
    import ipympl
    get_ipython().run_line_magic("matplotlib", "widget")
    _INTERACTIVE = True
    print("Backend: matplotlib 'widget' (hover enabled)")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")
    _INTERACTIVE = False
    print("Backend: 'inline' (hover disabled)")

import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt  # ← Matplotlib描画は使わないため未使用（残してもOK）
from pathlib import Path
import os

# hover (optional)（Plotlyはデフォルトでホバー対応のため未使用）
try:
    import mplcursors
    _HAS_MPLCURSORS = True
except Exception:
    _HAS_MPLCURSORS = False

import ipywidgets as W
from IPython.display import display, clear_output

# ★ Plotly を追加
import plotly.graph_objects as go

# ---------- file dialogs ----------
_USE_TK = True
try:
    import tkinter as tk
    from tkinter import filedialog
except Exception:
    _USE_TK = False

def open_file_dialog():
    if not _USE_TK:
        return None
    try:
        root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
        path = filedialog.askopenfilename(title="Open CSV",
                                          filetypes=[("CSV files","*.csv"),("All files","*.*")])
        root.destroy()
        return path or None
    except Exception:
        return None

def save_file_dialog(default_dir="", default_name="fft_result.csv"):
    if not _USE_TK:
        return None
    try:
        root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
        initialdir = default_dir if (default_dir and os.path.isdir(default_dir)) else os.getcwd()
        path = filedialog.asksaveasfilename(title="Save FFT CSV",
                                            defaultextension=".csv",
                                            initialdir=initialdir,
                                            initialfile=default_name,
                                            filetypes=[("CSV files","*.csv"),("All files","*.*")])
        root.destroy()
        return path or None
    except Exception:
        return None

# ---------- helpers ----------
def _normalize(c): return str(c).strip().lower()

def _is_monotonic_time_like(x, tol=1e-12):
    x = np.asarray(pd.to_numeric(x, errors="coerce"), float)
    dx = np.diff(x)
    return (dx.size > 0) and (np.sum(dx < -tol) == 0) and (np.median(dx) > tol)

def guess_columns(df: pd.DataFrame):
    cols = list(df.columns)
    nums = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    # time
    time_keys = ["time","seconds","time [s]","timestamp","時間","時刻","t"]
    time_col = None
    for c in nums:
        name = _normalize(c)
        if any(k in name for k in time_keys) and _is_monotonic_time_like(df[c]):
            time_col = c; break
    if time_col is None:
        monos = [c for c in nums if _is_monotonic_time_like(df[c])]
        if monos:
            best, best_j = None, None
            for c in monos:
                x = pd.to_numeric(df[c], errors="coerce").to_numpy(float)
                dt = np.diff(x); j = np.std(dt)/(np.mean(dt)+1e-20)
                if best is None or j < best_j: best, best_j = c, j
            time_col = best
    # ch1/ch2
    def _pick(keys, taken=set()):
        for c in nums:
            if c == time_col or c in taken: continue
            name = _normalize(c)
            if any(k in name for k in keys) and not _is_monotonic_time_like(df[c]):
                return c
        return None
    ch1_col = _pick(["ch1","ref","excitation","drive","channel1"])
    ch2_col = _pick(["ch2","rx","response","mag","channel2"], taken={ch1_col} if ch1_col else set())
    # 補完
    remain = [c for c in nums if c not in {time_col, ch1_col, ch2_col}]
    siglike = [c for c in remain if not _is_monotonic_time_like(df[c])]
    if ch1_col is None or ch2_col is None:
        scored = sorted(siglike, key=lambda c: float(np.nanstd(pd.to_numeric(df[c], errors="coerce"))), reverse=True)
        for c in scored:
            if ch1_col is None: ch1_col = c
            elif ch2_col is None and c != ch1_col: ch2_col = c; break
    return time_col, ch1_col, ch2_col

def infer_fs_from_time(t):
    t = np.asarray(t, float)
    if len(t) < 2: return None
    dt = np.median(np.diff(t))
    return None if dt <= 0 else 1.0/dt

def get_window(name, n):
    name = name.lower()
    if name == "hann":     return np.hanning(n)
    if name == "hamming":  return np.hamming(n)
    if name == "blackman": return np.blackman(n)
    return np.ones(n)

def next_pow2(n): return 1 << (int(np.ceil(np.log2(max(1, int(n))))))

def _wrap_deg(phi_deg):
    """-180..180 にラップ"""
    return (phi_deg + 180.0) % 360.0 - 180.0

# ---- FFT: Single（複素スペクトル & 振幅） ----
def single_fft_complex(x, fs, window="Hann", detrend=True, nfft_mode="Auto", custom_nfft=None, zpad=1):
    x = np.asarray(pd.to_numeric(x, errors="coerce"), float)
    if detrend: x = x - np.nanmean(x)
    x = np.nan_to_num(x, nan=0.0)
    n = len(x)
    if nfft_mode == "Next pow2": nfft = next_pow2(n)
    elif nfft_mode == "Custom" and custom_nfft and custom_nfft >= 1: nfft = int(custom_nfft)
    else: nfft = n
    nfft = int(nfft * max(1, int(zpad)))
    w = get_window(window, n)
    X = np.fft.rfft(x * w, n=nfft)
    f = np.fft.rfftfreq(nfft, 1.0/fs)
    cg = np.sum(w)/n
    A = np.abs(X) * (2.0/(n*cg))
    return f, X, A

# ---- FFT: Welch（複素平均 & パワー平均、クロス/オート）----
def welch_spectra(x, y, fs, window="Hann", detrend=True, seglen=16384, overlap=0.5,
                  nfft_mode="Auto", custom_nfft=None, zpad=1):
    """
    戻り値:
      f, Xavg, Yavg, Sxx, Syy, Syx, Ax, Ay
      - Xavg, Yavg: 複素スペクトルのコヒーレント平均（位相推定用）
      - Sxx, Syy, Syx: Welch平均のオート/クロススペクトル
      - Ax, Ay: 振幅（RMS換算の√パワー平均に対応する大きさ）
    """
    x = np.asarray(pd.to_numeric(x, errors="coerce"), float)
    y = np.asarray(pd.to_numeric(y, errors="coerce"), float)
    if detrend:
        x = x - np.nanmean(x); y = y - np.nanmean(y)
    x = np.nan_to_num(x, nan=0.0); y = np.nan_to_num(y, nan=0.0)
    n = len(x); L = int(max(8, min(seglen, n)))
    hop = max(1, int(L*(1-overlap)))
    if nfft_mode == "Next pow2": nfft = next_pow2(L)
    elif nfft_mode == "Custom" and custom_nfft and custom_nfft >= 1: nfft = int(custom_nfft)
    else: nfft = L
    nfft = int(nfft * max(1, int(zpad)))
    w = get_window(window, L)
    U = np.sum(w**2)  # 窓パワー

    acc_Sxx = acc_Syy = acc_Syx = None
    acc_powX = acc_powY = None
    acc_X = acc_Y = None
    k = 0
    for s in range(0, n-L+1, hop):
        xs = x[s:s+L]*w; ys = y[s:s+L]*w
        X = np.fft.rfft(xs, n=nfft); Y = np.fft.rfft(ys, n=nfft)
        Sxx_i = (X*np.conj(X)) / (U*fs)
        Syy_i = (Y*np.conj(Y)) / (U*fs)
        Syx_i = (Y*np.conj(X)) / (U*fs)
        acc_Sxx = Sxx_i if acc_Sxx is None else acc_Sxx + Sxx_i
        acc_Syy = Syy_i if acc_Syy is None else acc_Syy + Syy_i
        acc_Syx = Syx_i if acc_Syx is None else acc_Syx + Syx_i
        # 振幅のために |X|^2, |Y|^2 を蓄積
        acc_powX = (np.abs(X)**2) if acc_powX is None else acc_powX + (np.abs(X)**2)
        acc_powY = (np.abs(Y)**2) if acc_powY is None else acc_powY + (np.abs(Y)**2)
        # 位相用に複素平均
        acc_X = X if acc_X is None else acc_X + X
        acc_Y = Y if acc_Y is None else acc_Y + Y
        k += 1
    if k == 0:
        # fallback: single
        f, X, Ax = single_fft_complex(x, fs, window, detrend, nfft_mode, custom_nfft, zpad)
        _, Y, Ay = single_fft_complex(y, fs, window, detrend, nfft_mode, custom_nfft, zpad)
        Sxx = X*np.conj(X); Syy = Y*np.conj(Y); Syx = Y*np.conj(X)
        return f, X, Y, Sxx, Syy, Syx, Ax, Ay

    Sxx = acc_Sxx / k; Syy = acc_Syy / k; Syx = acc_Syx / k
    # 振幅は √(平均|X|^2)
    Ax = np.sqrt(acc_powX / k)
    Ay = np.sqrt(acc_powY / k)
    Xavg = acc_X / k; Yavg = acc_Y / k
    f = np.fft.rfftfreq(nfft, 1.0/fs)
    return f, Xavg, Yavg, Sxx, Syy, Syx, Ax, Ay

def attach_hover(line, xlabel, ylabel):
    # Plotly は標準で hover 表示があるため、ここは何もしません（互換用の空関数）
    return

# 追加：log軸用レンジ計算（数値→log10のレンジに変換）
def _log_range_from_linear(y0, y1, eps=1e-300):
    y0 = max(float(y0), eps)
    y1 = max(float(y1), y0 * (1.0 + 1e-12))
    return [np.log10(y0), np.log10(y1)]

# ====== make_box：Plotly版（logYで<1を確実に表示 & 最大値も反映） ======
def make_box(x, y, xlabel, ylabel, title, logy=False):
    x = np.asarray(x, float); y = np.asarray(y, float)

    # Xの初期レンジ
    xmin0 = float(np.nanmin(x)); xmax0 = float(np.nanmax(x))
    if xmin0 == xmax0: xmax0 = xmin0 + 1e-12

    # Yの初期レンジ（log時は正の値だけを対象）
    if logy:
        y_pos = y[np.isfinite(y) & (y > 0)]
        if y_pos.size == 0:
            y_pos = np.array([1e-12, 1.0])
        ymin0 = float(np.min(y_pos))
        ymax0 = float(np.max(y_pos))
        if ymin0 == ymax0:
            ymax0 = ymin0 * 10.0
        init_y_range = _log_range_from_linear(ymin0, ymax0)  # ★log10指定
    else:
        ymin0 = float(np.nanmin(y)); ymax0 = float(np.nanmax(y))
        if ymin0 == ymax0: ymax0 = ymin0 + 1e-12
        init_y_range = [ymin0, ymax0]

    # FigureWidget
    fig = go.FigureWidget(
        data=[go.Scattergl(
            x=x, y=y, mode="lines", line=dict(width=1),
            hovertemplate=f"{xlabel}: %{{x:.3e}}<br>{ylabel}: %{{y:.3e}}<extra></extra>"
        )],
        layout=dict(
            title=title,
            xaxis=dict(title=xlabel, range=[xmin0, xmax0]),
            yaxis=dict(
                title=ylabel,
                type=("log" if logy else "linear"),
                range=init_y_range  # ★初期から正しく設定
            ),
            template="plotly_white"
        )
    )

    # コントロール（元仕様踏襲）
    sx = W.FloatRangeSlider(value=[xmin0, xmax0], min=xmin0, max=xmax0,
                            step=max((xmax0-xmin0)/200, 1e-12),
                            description="xlim", continuous_update=True,
                            layout=W.Layout(width="60%"))
    sy = W.FloatRangeSlider(value=[ymin0, ymax0], min=ymin0, max=ymax0,
                            step=max((ymax0-ymin0)/200, 1e-12),
                            description="ylim", continuous_update=True,
                            layout=W.Layout(width="60%"))
    x_min = W.FloatText(value=xmin0, description="xmin", layout=W.Layout(width="19%"))
    x_max = W.FloatText(value=xmax0, description="xmax", layout=W.Layout(width="19%"))
    y_min = W.FloatText(value=ymin0, description="ymin", layout=W.Layout(width="19%"))
    y_max = W.FloatText(value=ymax0, description="ymax", layout=W.Layout(width="19%"))

    _lock = {"on": False}
    def _with_lock(fn):
        def wrapper(*args, **kwargs):
            if _lock["on"]: return
            _lock["on"] = True
            try: fn(*args, **kwargs)
            finally: _lock["on"] = False
        return wrapper

    def _apply_axes():
        fig.update_xaxes(range=[sx.value[0], sx.value[1]])
        y0, y1 = sy.value
        if logy:
            fig.update_yaxes(type="log", range=_log_range_from_linear(y0, y1))  # ★log10レンジで反映
        else:
            fig.update_yaxes(type="linear", range=[y0, y1])

    @_with_lock
    def _on_sx(_=None):
        x_min.value, x_max.value = sx.value; _apply_axes()
    @_with_lock
    def _on_sy(_=None):
        y_min.value, y_max.value = sy.value; _apply_axes()
    @_with_lock
    def _on_xnum(_=None):
        a, b = float(x_min.value), float(x_max.value)
        if a == b: b = a + 1e-12
        if a < sx.min: sx.min = a
        if b > sx.max: sx.max = b
        sx.value = [min(a,b), max(a,b)]; _apply_axes()
    @_with_lock
    def _on_ynum(_=None):
        a, b = float(y_min.value), float(y_max.value)
        if a == b: b = a + 1e-12
        if a < sy.min: sy.min = a
        if b > sy.max: sy.max = b
        sy.value = [min(a,b), max(a,b)]; _apply_axes()

    sx.observe(_on_sx, names="value"); sy.observe(_on_sy, names="value")
    x_min.observe(_on_xnum, names="value"); x_max.observe(_on_xnum, names="value")
    y_min.observe(_on_ynum, names="value"); y_max.observe(_on_ynum, names="value")

    # 初期適用
    _apply_axes()

    x_controls = W.HBox([sx, x_min, x_max])
    y_controls = W.HBox([sy, y_min, y_max])
    return W.VBox([fig, x_controls, y_controls])
# ---------- UI ----------
open_btn   = W.Button(description="Open CSV…", button_style="primary")
loaded_lbl = W.HTML("")
warn_tk    = W.HTML("" if _USE_TK else "<span style='color:#c00'>Note: tkinter not available; dialogs disabled.</span>")

method     = W.ToggleButtons(options=["Single FFT","Welch"], value="Single FFT", description="Method")
window_w   = W.Dropdown(options=["Rectangular","Hann","Hamming","Blackman"], value="Hann", description="Window")
detrend_w  = W.Checkbox(value=True, description="Remove DC")
nfft_mode  = W.Dropdown(options=["Auto","Next pow2","Custom"], value="Auto", description="NFFT")
custom_n   = W.IntText(value=131072, description="Custom NFFT")
zpad_w     = W.Dropdown(options=[1,2,4,8], value=1, description="Zero-pad×")
seglen_w   = W.IntText(value=16384, description="SegLen (Welch)")
overlap_w  = W.FloatSlider(value=0.5, min=0.0, max=0.9, step=0.1, readout_format='.1f', description="Overlap (Welch)")
logy_w     = W.Checkbox(value=True, description="Log Y (FFT)")
fmax_w     = W.FloatText(value=0.0, description="Fmax [Hz] (0=Nyq)")

# 位相モードとラップ設定
phase_mode = W.ToggleButtons(
    options=["Absolute (CH1 & CH2)", "Relative (CH2 − CH1)", "Transfer Function (CH2 wrt CH1)"],
    value="Transfer Function (CH2 wrt CH1)",
    description="Phase"
)
wrap_phase = W.Checkbox(value=True, description="Wrap phase to [−180, +180]")

def _toggle_custom(*args):
    custom_n.disabled = (nfft_mode.value != "Custom")
nfft_mode.observe(_toggle_custom, names="value"); _toggle_custom()

export_btn  = W.Button(description="Export FFT CSV…")
export_note = W.HTML("")

plots_box = W.VBox([])
_state = dict(
    df=None, t=None, ch1=None, ch2=None, fs=None,
    time_col=None, ch1_col=None, ch2_col=None,
    last_fft=None  # dict with computed arrays for export
)

def _compute_and_plot():
    st = _state
    if st["df"] is None: return
    t, x, y, fs = st["t"], st["ch1"], st["ch2"], st["fs"]

    # Compute spectra according to method
    if method.value == "Welch":
        f, Xavg, Yavg, Sxx, Syy, Syx, Ax, Ay = welch_spectra(
            x, y, fs, window=window_w.value, detrend=detrend_w.value,
            seglen=max(8, int(seglen_w.value)), overlap=float(overlap_w.value),
            nfft_mode=nfft_mode.value, custom_nfft=int(custom_n.value), zpad=int(zpad_w.value)
        )
        # Absolute phases from complex average
        P1 = np.unwrap(np.angle(Xavg)) * 180/np.pi
        P2 = np.unwrap(np.angle(Yavg)) * 180/np.pi
        # Relative phase and Transfer Function phase
        P_rel = np.unwrap(np.angle(Yavg) - np.angle(Xavg)) * 180/np.pi
        H = Syx / (Sxx + 1e-30)
        P_tf = np.unwrap(np.angle(H)) * 180/np.pi
        A1, A2 = Ax, Ay  # amplitude proxies (Welch)
    else:
        f1, X, A1 = single_fft_complex(x, fs, window=window_w.value, detrend=detrend_w.value,
                                       nfft_mode=nfft_mode.value, custom_nfft=int(custom_n.value), zpad=int(zpad_w.value))
        f2, Y, A2 = single_fft_complex(y, fs, window=window_w.value, detrend=detrend_w.value,
                                       nfft_mode=nfft_mode.value, custom_nfft=int(custom_n.value), zpad=int(zpad_w.value))
        # align
        n = min(len(f1), len(f2))
        f = f1[:n]; X = X[:n]; Y = Y[:n]; A1 = A1[:n]; A2 = A2[:n]
        P1 = np.unwrap(np.angle(X)) * 180/np.pi
        P2 = np.unwrap(np.angle(Y)) * 180/np.pi
        P_rel = np.unwrap(np.angle(Y) - np.angle(X)) * 180/np.pi
        H = Y / (X + 1e-30)
        P_tf = np.unwrap(np.angle(H)) * 180/np.pi

    nyq = fs/2.0
    fmax = float(fmax_w.value) if fmax_w.value and fmax_w.value > 0 else nyq
    idx = f <= fmax

    # phase wrapping if requested
    def _maybe_wrap(phi):
        return _wrap_deg(phi) if wrap_phase.value else phi

    # ----- build plots -----
    boxes = []
    # time + magnitude for CH1/CH2
    boxes.append(make_box(t, x, "Time [s]", "Amplitude", f"CH1 ({st['ch1_col']}) Time", logy=False))
    boxes.append(make_box(f[idx], A1[idx], "Frequency [Hz]", "Amplitude (approx.)",
                          f"CH1 ({st['ch1_col']}) FFT — {method.value}", logy=logy_w.value))
    boxes.append(make_box(t, y, "Time [s]", "Amplitude", f"CH2 ({st['ch2_col']}) Time", logy=False))
    boxes.append(make_box(f[idx], A2[idx], "Frequency [Hz]", "Amplitude (approx.)",
                          f"CH2 ({st['ch2_col']}) FFT — {method.value}", logy=logy_w.value))

    # phase plots depending on mode
    if phase_mode.value.startswith("Absolute"):
        boxes.append(make_box(f[idx], _maybe_wrap(P1[idx]), "Frequency [Hz]", "Phase [deg]",
                              f"CH1 ({st['ch1_col']}) Phase — {method.value}", logy=False))
        boxes.append(make_box(f[idx], _maybe_wrap(P2[idx]), "Frequency [Hz]", "Phase [deg]",
                              f"CH2 ({st['ch2_col']}) Phase — {method.value}", logy=False))
        phase_summary = "Absolute phases (CH1/CH2)"
    elif phase_mode.value.startswith("Relative"):
        boxes.append(make_box(f[idx], _maybe_wrap(P_rel[idx]), "Frequency [Hz]", "Phase [deg]",
                              f"Relative Phase (CH2 − CH1) — {method.value}", logy=False))
        phase_summary = "Relative phase (CH2−CH1)"
    else:
        boxes.append(make_box(f[idx], _maybe_wrap(P_tf[idx]), "Frequency [Hz]", "Phase [deg]",
                              f"Transfer Function Phase H=CH2/CH1 — {method.value}", logy=False))
        phase_summary = "Transfer Function phase"

    # layout
    children = [
        W.HTML("<b>CH1 (Excitation)</b>"), boxes[0], boxes[1],
        W.HTML("<b>CH2 (Magnetization)</b>"), boxes[2], boxes[3],
        W.HTML(f"<b>Phase — {phase_summary}</b>")
    ] + boxes[4:]
    plots_box.children = children

    # save for export
    _exp = dict(
        f=f, A1=A1, A2=A2, P1=P1, P2=P2, P_rel=P_rel, P_tf=P_tf,
        mode=method.value, wrap=wrap_phase.value
    )
    _state["last_fft"] = _exp

def on_open_clicked(_):
    path = open_file_dialog()
    if not path:
        if _USE_TK:
            loaded_lbl.value = "<span style='color:#c60'>Canceled or no file selected.</span>"
        else:
            loaded_lbl.value = "<span style='color:#c00'>tkinter not available; dialogs disabled.</span>"
        return
    p = Path(path)
    try:
        df = pd.read_csv(p, comment="#", engine="python", sep=None).dropna(axis=1, how="all")
    except Exception as e:
        loaded_lbl.value = f"<span style='color:#c00'>Failed to read CSV: {e}</span>"
        return

    time_col, ch1_col, ch2_col = guess_columns(df)
    if (ch1_col is None) or (ch2_col is None):
        loaded_lbl.value = "<span style='color:#c00'>CH1/CH2 columns not found. Check CSV.</span>"
        return

    if (time_col is not None) and pd.api.types.is_numeric_dtype(df[time_col]) and _is_monotonic_time_like(df[time_col]):
        t = pd.to_numeric(df[time_col], errors="coerce").to_numpy(float)
        fs = infer_fs_from_time(t)
        if fs is None or not np.isfinite(fs): fs = 1_000_000.0
        src = f"time column '{time_col}'"
    else:
        fs = 1_000_000.0
        t  = np.arange(len(df)) / fs
        src = "generated fs=1e6 Hz"

    ch1 = pd.to_numeric(df[ch1_col], errors="coerce").to_numpy(float)
    ch2 = pd.to_numeric(df[ch2_col], errors="coerce").to_numpy(float)
    n = min(len(t), len(ch1), len(ch2))
    t, ch1, ch2 = t[:n], ch1[:n], ch2[:n]

    _state.update(dict(df=df, t=t, ch1=ch1, ch2=ch2, fs=float(fs),
                       time_col=time_col, ch1_col=ch1_col, ch2_col=ch2_col, last_fft=None))
    loaded_lbl.value = (f"<span style='color:green'>Loaded: {p.name} | samples={n}, fs={fs:.3f} Hz, "
                        f"t=[{t.min():.6f}, {t.max():.6f}] s ({src})</span>")

    fmax_w.value = float(fs/2.0)
    _compute_and_plot()

def on_any_change(_):
    if _state["df"] is None: return
    _compute_and_plot()

def on_export_clicked(_):
    st = _state
    exp = st.get("last_fft", None)
    if exp is None:
        export_note.value = "<span style='color:#c00'>No FFT yet. Open a CSV first.</span>"
        return
    default_dir = os.getcwd()
    save_path = save_file_dialog(default_dir=default_dir, default_name="fft_result.csv")
    if not save_path:
        if _USE_TK:
            export_note.value = "<span style='color:#c60'>Canceled or no path selected.</span>"
        else:
            export_note.value = "<span style='color:#c00'>tkinter not available. Cannot open save dialog.</span>"
        return

    f = exp["f"]; A1 = exp["A1"]; A2 = exp["A2"]
    # export both wrapped and unwrapped variants for phases
    P1 = exp["P1"]; P2 = exp["P2"]; P_rel = exp["P_rel"]; P_tf = exp["P_tf"]
    P1w = _wrap_deg(P1); P2w = _wrap_deg(P2); P_relw = _wrap_deg(P_rel); P_tfw = _wrap_deg(P_tf)

    df_out = pd.DataFrame({
        "f_Hz": f,
        "CH1_amp": A1, "CH2_amp": A2,
        "CH1_phase_deg": P1, "CH1_phase_wrapped_deg": P1w,
        "CH2_phase_deg": P2, "CH2_phase_wrapped_deg": P2w,
        "Rel_phase_deg": P_rel, "Rel_phase_wrapped_deg": P_relw,
        "TF_phase_deg": P_tf, "TF_phase_wrapped_deg": P_tfw
    })
    try:
        df_out.to_csv(save_path, index=False)
        export_note.value = f"<span style='color:green'>Saved: {save_path}</span>"
    except Exception as e:
        export_note.value = f"<span style='color:#c00'>Save failed: {e}</span>"

open_btn.on_click(on_open_clicked)
export_btn.on_click(on_export_clicked)

for w in [method, window_w, detrend_w, nfft_mode, custom_n, zpad_w,
          seglen_w, overlap_w, logy_w, fmax_w, phase_mode, wrap_phase]:
    w.observe(on_any_change, names="value")

# layout
top_bar = W.HBox([open_btn, loaded_lbl, warn_tk])
display(top_bar)
controls_left  = W.VBox([method, window_w, detrend_w, logy_w])
controls_mid   = W.VBox([phase_mode, wrap_phase])
controls_right = W.VBox([nfft_mode, custom_n, zpad_w, seglen_w, overlap_w, fmax_w])
display(W.HBox([controls_left, controls_mid, controls_right]))
display(plots_box)
display(W.HTML("<hr><b>Export</b>"))
display(W.HBox([export_btn, export_note]))


Backend: matplotlib 'widget' (hover enabled)


VBox()

HTML(value='<hr><b>Export</b>')

In [7]:
# ==== Oscilloscope Analyzer — class-based (Plotly) ====

import numpy as np
import pandas as pd
from pathlib import Path
import os

import ipywidgets as W
from IPython.display import display, clear_output

# Plotly
import plotly.graph_objects as go


class OscilloscopeAnalyzer:
    """Encapsulated UI + logic. Call display() to render in Jupyter."""

    # ---------- static helpers ----------
    @staticmethod
    def _normalize(c): return str(c).strip().lower()

    @staticmethod
    def _is_monotonic_time_like(x, tol=1e-12):
        x = np.asarray(pd.to_numeric(x, errors="coerce"), float)
        dx = np.diff(x)
        return (dx.size > 0) and (np.sum(dx < -tol) == 0) and (np.median(dx) > tol)

    @staticmethod
    def infer_fs_from_time(t):
        t = np.asarray(t, float)
        if len(t) < 2: return None
        dt = np.median(np.diff(t))
        return None if dt <= 0 else 1.0/dt

    @staticmethod
    def get_window(name, n):
        name = name.lower()
        if name == "hann":     return np.hanning(n)
        if name == "hamming":  return np.hamming(n)
        if name == "blackman": return np.blackman(n)
        return np.ones(n)

    @staticmethod
    def next_pow2(n): return 1 << (int(np.ceil(np.log2(max(1, int(n))))))

    @staticmethod
    def _wrap_deg(phi_deg):
        """-180..180 wrap"""
        return (phi_deg + 180.0) % 360.0 - 180.0

    @staticmethod
    def _log_range_from_linear(y0, y1, eps=1e-300):
        y0 = max(float(y0), eps)
        y1 = max(float(y1), y0 * (1.0 + 1e-12))
        return [np.log10(y0), np.log10(y1)]

    # ---------- CSV column guess ----------
    @classmethod
    def guess_columns(cls, df: pd.DataFrame):
        cols = list(df.columns)
        nums = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
        # time
        time_keys = ["time", "seconds", "time [s]", "timestamp", "時間", "時刻", "t"]
        time_col = None
        for c in nums:
            name = cls._normalize(c)
            if any(k in name for k in time_keys) and cls._is_monotonic_time_like(df[c]):
                time_col = c; break
        if time_col is None:
            monos = [c for c in nums if cls._is_monotonic_time_like(df[c])]
            if monos:
                best, best_j = None, None
                for c in monos:
                    x = pd.to_numeric(df[c], errors="coerce").to_numpy(float)
                    dt = np.diff(x); j = np.std(dt)/(np.mean(dt)+1e-20)
                    if best is None or j < best_j: best, best_j = c, j
                time_col = best
        # ch1/ch2
        def _pick(keys, taken=set()):
            for c in nums:
                if c == time_col or c in taken: continue
                name = cls._normalize(c)
                if any(k in name for k in keys) and not cls._is_monotonic_time_like(df[c]):
                    return c
            return None
        ch1_col = _pick(["ch1","ref","excitation","drive","channel1"])
        ch2_col = _pick(["ch2","rx","response","mag","channel2"], taken={ch1_col} if ch1_col else set())
        # fill
        remain = [c for c in nums if c not in {time_col, ch1_col, ch2_col}]
        siglike = [c for c in remain if not cls._is_monotonic_time_like(df[c])]
        if ch1_col is None or ch2_col is None:
            scored = sorted(siglike, key=lambda c: float(np.nanstd(pd.to_numeric(df[c], errors="coerce"))), reverse=True)
            for c in scored:
                if ch1_col is None: ch1_col = c
                elif ch2_col is None and c != ch1_col: ch2_col = c; break
        return time_col, ch1_col, ch2_col

    # ---------- FFT ----------
    @classmethod
    def single_fft_complex(cls, x, fs, window="Hann", detrend=True, nfft_mode="Auto", custom_nfft=None, zpad=1):
        x = np.asarray(pd.to_numeric(x, errors="coerce"), float)
        if detrend: x = x - np.nanmean(x)
        x = np.nan_to_num(x, nan=0.0)
        n = len(x)
        if nfft_mode == "Next pow2": nfft = cls.next_pow2(n)
        elif nfft_mode == "Custom" and custom_nfft and custom_nfft >= 1: nfft = int(custom_nfft)
        else: nfft = n
        nfft = int(nfft * max(1, int(zpad)))
        w = cls.get_window(window, n)
        X = np.fft.rfft(x * w, n=nfft)
        f = np.fft.rfftfreq(nfft, 1.0/fs)
        cg = np.sum(w)/n
        A = np.abs(X) * (2.0/(n*cg))
        return f, X, A

    @classmethod
    def welch_spectra(cls, x, y, fs, window="Hann", detrend=True, seglen=16384, overlap=0.5,
                      nfft_mode="Auto", custom_nfft=None, zpad=1):
        """
        Returns:
          f, Xavg, Yavg, Sxx, Syy, Syx, Ax, Ay
        """
        x = np.asarray(pd.to_numeric(x, errors="coerce"), float)
        y = np.asarray(pd.to_numeric(y, errors="coerce"), float)
        if detrend:
            x = x - np.nanmean(x); y = y - np.nanmean(y)
        x = np.nan_to_num(x, nan=0.0); y = np.nan_to_num(y, nan=0.0)
        n = len(x); L = int(max(8, min(seglen, n)))
        hop = max(1, int(L*(1-overlap)))
        if nfft_mode == "Next pow2": nfft = cls.next_pow2(L)
        elif nfft_mode == "Custom" and custom_nfft and custom_nfft >= 1: nfft = int(custom_nfft)
        else: nfft = L
        nfft = int(nfft * max(1, int(zpad)))
        w = cls.get_window(window, L)
        U = np.sum(w**2)

        acc_Sxx = acc_Syy = acc_Syx = None
        acc_powX = acc_powY = None
        acc_X = acc_Y = None
        k = 0
        for s in range(0, n-L+1, hop):
            xs = x[s:s+L]*w; ys = y[s:s+L]*w
            X = np.fft.rfft(xs, n=nfft); Y = np.fft.rfft(ys, n=nfft)
            Sxx_i = (X*np.conj(X)) / (U*fs)
            Syy_i = (Y*np.conj(Y)) / (U*fs)
            Syx_i = (Y*np.conj(X)) / (U*fs)
            acc_Sxx = Sxx_i if acc_Sxx is None else acc_Sxx + Sxx_i
            acc_Syy = Syy_i if acc_Syy is None else acc_Syy + Syy_i
            acc_Syx = Syx_i if acc_Syx is None else acc_Syx + Syx_i
            acc_powX = (np.abs(X)**2) if acc_powX is None else acc_powX + (np.abs(X)**2)
            acc_powY = (np.abs(Y)**2) if acc_powY is None else acc_powY + (np.abs(Y)**2)
            acc_X = X if acc_X is None else acc_X + X
            acc_Y = Y if acc_Y is None else acc_Y + Y
            k += 1
        if k == 0:
            f, X, Ax = cls.single_fft_complex(x, fs, window, detrend, nfft_mode, custom_nfft, zpad)
            _, Y, Ay = cls.single_fft_complex(y, fs, window, detrend, nfft_mode, custom_nfft, zpad)
            Sxx = X*np.conj(X); Syy = Y*np.conj(Y); Syx = Y*np.conj(X)
            return f, X, Y, Sxx, Syy, Syx, Ax, Ay

        Sxx = acc_Sxx / k; Syy = acc_Syy / k; Syx = acc_Syx / k
        Ax = np.sqrt(acc_powX / k)
        Ay = np.sqrt(acc_powY / k)
        Xavg = acc_X / k; Yavg = acc_Y / k
        f = np.fft.rfftfreq(nfft, 1.0/fs)
        return f, Xavg, Yavg, Sxx, Syy, Syx, Ax, Ay

    # ---------- dialogs ----------
    @staticmethod
    def _tk_available():
        try:
            import tkinter as tk  # noqa
            return True
        except Exception:
            return False

    @staticmethod
    def open_file_dialog():
        if not OscilloscopeAnalyzer._tk_available():
            return None
        try:
            import tkinter as tk
            from tkinter import filedialog
            root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
            path = filedialog.askopenfilename(title="Open CSV",
                                              filetypes=[("CSV files","*.csv"),("All files","*.*")])
            root.destroy()
            return path or None
        except Exception:
            return None

    @staticmethod
    def save_file_dialog(default_dir="", default_name="fft_result.csv"):
        if not OscilloscopeAnalyzer._tk_available():
            return None
        try:
            import tkinter as tk
            from tkinter import filedialog
            root = tk.Tk(); root.withdraw(); root.attributes("-topmost", True)
            initialdir = default_dir if (default_dir and os.path.isdir(default_dir)) else os.getcwd()
            path = filedialog.asksaveasfilename(title="Save FFT CSV",
                                                defaultextension=".csv",
                                                initialdir=initialdir,
                                                initialfile=default_name,
                                                filetypes=[("CSV files","*.csv"),("All files","*.*")])
            root.destroy()
            return path or None
        except Exception:
            return None

    # ---------- UI construction ----------
    def __init__(self):
        # widgets
        self.open_btn   = W.Button(description="Open CSV…", button_style="primary")
        self.loaded_lbl = W.HTML("")
        self.warn_tk    = W.HTML("" if self._tk_available() else "<span style='color:#c00'>Note: tkinter not available; dialogs disabled.</span>")

        self.method     = W.ToggleButtons(options=["Single FFT","Welch"], value="Single FFT", description="Method")
        self.window_w   = W.Dropdown(options=["Rectangular","Hann","Hamming","Blackman"], value="Hann", description="Window")
        self.detrend_w  = W.Checkbox(value=True, description="Remove DC")
        self.nfft_mode  = W.Dropdown(options=["Auto","Next pow2","Custom"], value="Auto", description="NFFT")
        self.custom_n   = W.IntText(value=131072, description="Custom NFFT")
        self.zpad_w     = W.Dropdown(options=[1,2,4,8], value=1, description="Zero-pad×")
        self.seglen_w   = W.IntText(value=16384, description="SegLen (Welch)")
        self.overlap_w  = W.FloatSlider(value=0.5, min=0.0, max=0.9, step=0.1, readout_format='.1f', description="Overlap (Welch)")
        self.logy_w     = W.Checkbox(value=True, description="Log Y (FFT)")
        self.fmax_w     = W.FloatText(value=0.0, description="Fmax [Hz] (0=Nyq)")

        self.phase_mode = W.ToggleButtons(
            options=["Absolute (CH1 & CH2)", "Relative (CH2 − CH1)", "Transfer Function (CH2 wrt CH1)"],
            value="Transfer Function (CH2 wrt CH1)", description="Phase")
        self.wrap_phase = W.Checkbox(value=True, description="Wrap phase to [−180, +180]")

        self.export_btn  = W.Button(description="Export FFT CSV…")
        self.export_note = W.HTML("")

        self.plots_box = W.VBox([])

        # state
        self.state = dict(
            df=None, t=None, ch1=None, ch2=None, fs=None,
            time_col=None, ch1_col=None, ch2_col=None,
            last_fft=None
        )

        # wiring
        self.open_btn.on_click(self._on_open_clicked)
        self.export_btn.on_click(self._on_export_clicked)

        for w in [self.method, self.window_w, self.detrend_w, self.nfft_mode, self.custom_n, self.zpad_w,
                  self.seglen_w, self.overlap_w, self.logy_w, self.fmax_w, self.phase_mode, self.wrap_phase]:
            w.observe(self._on_any_change, names="value")

        # NFFT custom enable/disable
        self.nfft_mode.observe(self._toggle_custom, names="value")
        self._toggle_custom()

    # ---------- UI parts ----------
    def _toggle_custom(self, *_):
        self.custom_n.disabled = (self.nfft_mode.value != "Custom")

    def _make_box(self, x, y, xlabel, ylabel, title, logy=False):
        x = np.asarray(x, float); y = np.asarray(y, float)

        xmin0 = float(np.nanmin(x)); xmax0 = float(np.nanmax(x))
        if xmin0 == xmax0: xmax0 = xmin0 + 1e-12

        if logy:
            y_pos = y[np.isfinite(y) & (y > 0)]
            if y_pos.size == 0: y_pos = np.array([1e-12, 1.0])
            ymin0 = float(np.min(y_pos)); ymax0 = float(np.max(y_pos))
            if ymin0 == ymax0: ymax0 = ymin0 * 10.0
            init_y_range = self._log_range_from_linear(ymin0, ymax0)
        else:
            ymin0 = float(np.nanmin(y)); ymax0 = float(np.nanmax(y))
            if ymin0 == ymax0: ymax0 = ymin0 + 1e-12
            init_y_range = [ymin0, ymax0]

        fig = go.FigureWidget(
            data=[go.Scattergl(
                x=x, y=y, mode="lines", line=dict(width=1),
                hovertemplate=f"{xlabel}: %{{x:.3e}}<br>{ylabel}: %{{y:.3e}}<extra></extra>"
            )],
            layout=dict(
                title=title,
                xaxis=dict(title=xlabel, range=[xmin0, xmax0]),
                yaxis=dict(title=ylabel, type=("log" if logy else "linear"), range=init_y_range),
                template="plotly_white"
            )
        )

        sx = W.FloatRangeSlider(value=[xmin0, xmax0], min=xmin0, max=xmax0,
                                step=max((xmax0-xmin0)/200, 1e-12),
                                description="xlim", continuous_update=True,
                                layout=W.Layout(width="60%"))
        sy = W.FloatRangeSlider(value=[ymin0, ymax0], min=ymin0, max=ymax0,
                                step=max((ymax0-ymin0)/200, 1e-12),
                                description="ylim", continuous_update=True,
                                layout=W.Layout(width="60%"))
        x_min = W.FloatText(value=xmin0, description="xmin", layout=W.Layout(width="19%"))
        x_max = W.FloatText(value=xmax0, description="xmax", layout=W.Layout(width="19%"))
        y_min = W.FloatText(value=ymin0, description="ymin", layout=W.Layout(width="19%"))
        y_max = W.FloatText(value=ymax0, description="ymax", layout=W.Layout(width="19%"))

        lock = {"on": False}
        def _with_lock(fn):
            def wrapper(*args, **kwargs):
                if lock["on"]: return
                lock["on"] = True
                try: fn(*args, **kwargs)
                finally: lock["on"] = False
            return wrapper

        def _apply_axes():
            fig.update_xaxes(range=[sx.value[0], sx.value[1]])
            y0, y1 = sy.value
            if logy:
                fig.update_yaxes(type="log", range=self._log_range_from_linear(y0, y1))
            else:
                fig.update_yaxes(type="linear", range=[y0, y1])

        @_with_lock
        def _on_sx(_=None):
            x_min.value, x_max.value = sx.value; _apply_axes()
        @_with_lock
        def _on_sy(_=None):
            y_min.value, y_max.value = sy.value; _apply_axes()
        @_with_lock
        def _on_xnum(_=None):
            a, b = float(x_min.value), float(x_max.value)
            if a == b: b = a + 1e-12
            if a < sx.min: sx.min = a
            if b > sx.max: sx.max = b
            sx.value = [min(a,b), max(a,b)]; _apply_axes()
        @_with_lock
        def _on_ynum(_=None):
            a, b = float(y_min.value), float(y_max.value)
            if a == b: b = a + 1e-12
            if a < sy.min: sy.min = a
            if b > sy.max: sy.max = b
            sy.value = [min(a,b), max(a,b)]; _apply_axes()

        sx.observe(_on_sx, names="value"); sy.observe(_on_sy, names="value")
        x_min.observe(_on_xnum, names="value"); x_max.observe(_on_xnum, names="value")
        y_min.observe(_on_ynum, names="value"); y_max.observe(_on_ynum, names="value")

        _apply_axes()
        x_controls = W.HBox([sx, x_min, x_max])
        y_controls = W.HBox([sy, y_min, y_max])
        return W.VBox([fig, x_controls, y_controls])

    # ---------- compute/plot ----------
    def _compute_and_plot(self):
        st = self.state
        if st["df"] is None: return
        t, x, y, fs = st["t"], st["ch1"], st["ch2"], st["fs"]

        if self.method.value == "Welch":
            f, Xavg, Yavg, Sxx, Syy, Syx, Ax, Ay = self.welch_spectra(
                x, y, fs, window=self.window_w.value, detrend=self.detrend_w.value,
                seglen=max(8, int(self.seglen_w.value)), overlap=float(self.overlap_w.value),
                nfft_mode=self.nfft_mode.value, custom_nfft=int(self.custom_n.value), zpad=int(self.zpad_w.value)
            )
            P1 = np.unwrap(np.angle(Xavg)) * 180/np.pi
            P2 = np.unwrap(np.angle(Yavg)) * 180/np.pi
            P_rel = np.unwrap(np.angle(Yavg) - np.angle(Xavg)) * 180/np.pi
            H = Syx / (Sxx + 1e-30)
            P_tf = np.unwrap(np.angle(H)) * 180/np.pi
            A1, A2 = Ax, Ay
        else:
            f1, X, A1 = self.single_fft_complex(x, fs, window=self.window_w.value, detrend=self.detrend_w.value,
                                                nfft_mode=self.nfft_mode.value, custom_nfft=int(self.custom_n.value), zpad=int(self.zpad_w.value))
            f2, Y, A2 = self.single_fft_complex(y, fs, window=self.window_w.value, detrend=self.detrend_w.value,
                                                nfft_mode=self.nfft_mode.value, custom_nfft=int(self.custom_n.value), zpad=int(self.zpad_w.value))
            n = min(len(f1), len(f2))
            f = f1[:n]; X = X[:n]; Y = Y[:n]; A1 = A1[:n]; A2 = A2[:n]
            P1 = np.unwrap(np.angle(X)) * 180/np.pi
            P2 = np.unwrap(np.angle(Y)) * 180/np.pi
            P_rel = np.unwrap(np.angle(Y) - np.angle(X)) * 180/np.pi
            H = Y / (X + 1e-30)
            P_tf = np.unwrap(np.angle(H)) * 180/np.pi

        nyq = fs/2.0
        fmax = float(self.fmax_w.value) if self.fmax_w.value and self.fmax_w.value > 0 else nyq
        idx = f <= fmax

        def _maybe_wrap(phi):
            return self._wrap_deg(phi) if self.wrap_phase.value else phi

        boxes = []
        boxes.append(self._make_box(t, x, "Time [s]", "Amplitude", f"CH1 ({st['ch1_col']}) Time", logy=False))
        boxes.append(self._make_box(f[idx], A1[idx], "Frequency [Hz]", "Amplitude (approx.)",
                                    f"CH1 ({st['ch1_col']}) FFT — {self.method.value}", logy=self.logy_w.value))
        boxes.append(self._make_box(t, y, "Time [s]", "Amplitude", f"CH2 ({st['ch2_col']}) Time", logy=False))
        boxes.append(self._make_box(f[idx], A2[idx], "Frequency [Hz]", "Amplitude (approx.)",
                                    f"CH2 ({st['ch2_col']}) FFT — {self.method.value}", logy=self.logy_w.value))

        if self.phase_mode.value.startswith("Absolute"):
            boxes.append(self._make_box(f[idx], _maybe_wrap(P1[idx]), "Frequency [Hz]", "Phase [deg]",
                                        f"CH1 ({st['ch1_col']}) Phase — {self.method.value}", logy=False))
            boxes.append(self._make_box(f[idx], _maybe_wrap(P2[idx]), "Frequency [Hz]", "Phase [deg]",
                                        f"CH2 ({st['ch2_col']}) Phase — {self.method.value}", logy=False))
            phase_summary = "Absolute phases (CH1/CH2)"
        elif self.phase_mode.value.startswith("Relative"):
            boxes.append(self._make_box(f[idx], _maybe_wrap(P_rel[idx]), "Frequency [Hz]", "Phase [deg]",
                                        f"Relative Phase (CH2 − CH1) — {self.method.value}", logy=False))
            phase_summary = "Relative phase (CH2−CH1)"
        else:
            boxes.append(self._make_box(f[idx], _maybe_wrap(P_tf[idx]), "Frequency [Hz]", "Phase [deg]",
                                        f"Transfer Function Phase H=CH2/CH1 — {self.method.value}", logy=False))
            phase_summary = "Transfer Function phase"

        children = [
            W.HTML("<b>CH1 (Excitation)</b>"), boxes[0], boxes[1],
            W.HTML("<b>CH2 (Magnetization)</b>"), boxes[2], boxes[3],
            W.HTML(f"<b>Phase — {phase_summary}</b>")
        ] + boxes[4:]
        self.plots_box.children = children

        self.state["last_fft"] = dict(
            f=f, A1=A1, A2=A2, P1=P1, P2=P2, P_rel=P_rel, P_tf=P_tf,
            mode=self.method.value, wrap=self.wrap_phase.value
        )

    # ---------- event handlers ----------
    def _on_open_clicked(self, _):
        path = self.open_file_dialog()
        if not path:
            if self._tk_available():
                self.loaded_lbl.value = "<span style='color:#c60'>Canceled or no file selected.</span>"
            else:
                self.loaded_lbl.value = "<span style='color:#c00'>tkinter not available; dialogs disabled.</span>"
            return
        p = Path(path)
        try:
            df = pd.read_csv(p, comment="#", engine="python", sep=None).dropna(axis=1, how="all")
        except Exception as e:
            self.loaded_lbl.value = f"<span style='color:#c00'>Failed to read CSV: {e}</span>"
            return

        time_col, ch1_col, ch2_col = self.guess_columns(df)
        if (ch1_col is None) or (ch2_col is None):
            self.loaded_lbl.value = "<span style='color:#c00'>CH1/CH2 columns not found. Check CSV.</span>"
            return

        if (time_col is not None) and pd.api.types.is_numeric_dtype(df[time_col]) and self._is_monotonic_time_like(df[time_col]):
            t = pd.to_numeric(df[time_col], errors="coerce").to_numpy(float)
            fs = self.infer_fs_from_time(t)
            if fs is None or not np.isfinite(fs): fs = 1_000_000.0
            src = f"time column '{time_col}'"
        else:
            fs = 1_000_000.0
            t  = np.arange(len(df)) / fs
            src = "generated fs=1e6 Hz"

        ch1 = pd.to_numeric(df[ch1_col], errors="coerce").to_numpy(float)
        ch2 = pd.to_numeric(df[ch2_col], errors="coerce").to_numpy(float)
        n = min(len(t), len(ch1), len(ch2))
        t, ch1, ch2 = t[:n], ch1[:n], ch2[:n]

        self.state.update(dict(df=df, t=t, ch1=ch1, ch2=ch2, fs=float(fs),
                               time_col=time_col, ch1_col=ch1_col, ch2_col=ch2_col, last_fft=None))
        self.loaded_lbl.value = (f"<span style='color:green'>Loaded: {p.name} | samples={n}, fs={fs:.3f} Hz, "
                                 f"t=[{t.min():.6f}, {t.max():.6f}] s ({src})</span>")
        self.fmax_w.value = float(fs/2.0)
        self._compute_and_plot()

    def _on_any_change(self, _):
        if self.state["df"] is None: return
        self._compute_and_plot()

    def _on_export_clicked(self, _):
        st = self.state
        exp = st.get("last_fft", None)
        if exp is None:
            self.export_note.value = "<span style='color:#c00'>No FFT yet. Open a CSV first.</span>"
            return
        default_dir = os.getcwd()
        save_path = self.save_file_dialog(default_dir=default_dir, default_name="fft_result.csv")
        if not save_path:
            if self._tk_available():
                self.export_note.value = "<span style='color:#c60'>Canceled or no path selected.</span>"
            else:
                self.export_note.value = "<span style='color:#c00'>tkinter not available. Cannot open save dialog.</span>"
            return

        f = exp["f"]; A1 = exp["A1"]; A2 = exp["A2"]
        P1 = exp["P1"]; P2 = exp["P2"]; P_rel = exp["P_rel"]; P_tf = exp["P_tf"]
        P1w = self._wrap_deg(P1); P2w = self._wrap_deg(P2); P_relw = self._wrap_deg(P_rel); P_tfw = self._wrap_deg(P_tf)

        df_out = pd.DataFrame({
            "f_Hz": f,
            "CH1_amp": A1, "CH2_amp": A2,
            "CH1_phase_deg": P1, "CH1_phase_wrapped_deg": P1w,
            "CH2_phase_deg": P2, "CH2_phase_wrapped_deg": P2w,
            "Rel_phase_deg": P_rel, "Rel_phase_wrapped_deg": P_relw,
            "TF_phase_deg": P_tf, "TF_phase_wrapped_deg": P_tfw
        })
        try:
            df_out.to_csv(save_path, index=False)
            self.export_note.value = f"<span style='color:green'>Saved: {save_path}</span>"
        except Exception as e:
            self.export_note.value = f"<span style='color:#c00'>Save failed: {e}</span>"

    # ---------- public render ----------
    def display(self):
        top_bar = W.HBox([self.open_btn, self.loaded_lbl, self.warn_tk])
        controls_left  = W.VBox([self.method, self.window_w, self.detrend_w, self.logy_w])
        controls_mid   = W.VBox([self.phase_mode, self.wrap_phase])
        controls_right = W.VBox([self.nfft_mode, self.custom_n, self.zpad_w, self.seglen_w, self.overlap_w, self.fmax_w])

        display(top_bar)
        display(W.HBox([controls_left, controls_mid, controls_right]))
        display(self.plots_box)
        display(W.HTML("<hr><b>Export</b>"))
        display(W.HBox([self.export_btn, self.export_note]))


# ---- launch guard (only executed when run, not on import) ----
if __name__ == "__main__":
    # In notebooks, this cell will run and render UI.
    app = OscilloscopeAnalyzer()
    app.display()


VBox()

HTML(value='<hr><b>Export</b>')